# 3. Inference and Visualisation

Using the model trained previously, this notebook performs slide-level inference by tiling the whole slide, scoring each patch, and stitching the results into a heatmap.

In [ ]:
import sys
sys.path.append('../src')
from pathlib import Path
import torch
import matplotlib.pyplot as plt
from tvs.data.transforms import build_transforms
from tvs.models.cnn_baseline import CNNBaseline
from tvs.inference.tiler import tile_wsi
from tvs.inference.stitcher import stitch_scores
from tvs.io.annotations import geojson_to_sqlite
from tiatoolbox.wsicore.wsireader import WSIReader

# Paths
data_dir = Path('../tests/test_data')
slide_path = data_dir / 'test_slide.svs'
db_path = data_dir / 'test_slide.db'
model_path = data_dir / 'toy_model.pt'

# Ensure annotation DB exists
geojson_path = data_dir / 'test_slide.geojson'
geojson_to_sqlite(geojson_path, db_path, cache=False)

# Load model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CNNBaseline().to(device)
state = torch.load(model_path, map_location=device)
model.load_state_dict(state['model'])
model.eval()

transform = build_transforms(train=False)

wsi = WSIReader.open(slide_path)
coords, scores = [], []
for coord, patch in tile_wsi(wsi, annotation_store=str(db_path), patch_size=512, stride=512, resolution=1.0, units='mpp'):
    img = transform(image=patch)['image'].unsqueeze(0).to(device)
    with torch.no_grad():
        score = model(img).item()
    coords.append(coord)
    scores.append(score)

thumb = wsi.slide_thumbnail(6.0, 'mpp')
heatmap = stitch_scores(coords, scores, thumb.shape[:2], scale=1.0/6.0)
plt.figure(figsize=(5,5))
plt.imshow(heatmap, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(label='score')
plt.title('Predicted score heatmap')
plt.axis('off')
plt.show()

print('Processed', len(scores), 'patches')